# CheXpert — Train DenseNet121 (Kaggle GPU)

Trains a multi-label thoracic disease classifier on **CheXpert-small** and exports
weights in the **handoff contract** the Mac app expects:

```
state_dict.pth      trained DenseNet121 weights
labels.json         label order (list[str])
model_config.json   {arch, input_size, num_classes, norm, sigmoid}
```

Download those three files from `/kaggle/working` at the end and drop them into
`medical-ai/models/checkpoints/`, then set `MODEL_SOURCE=local` on the Mac.

**Run this on Kaggle** with a GPU accelerator and the CheXpert-small dataset attached
(Add Input → search "chexpert"). `/kaggle/input` is read-only; we write to `/kaggle/working`.

### The uncertainty-label gotcha
CheXpert labels are `1.0` (positive), `0.0` (negative), `-1.0` (uncertain), or blank.
How you map `-1` materially changes AUROC. We use **U-Ones** (treat uncertain as positive)
for the atelectasis/edema-type labels where it helps, and **U-Zeros** elsewhere — configurable below.

In [ ]:
# --- Config -----------------------------------------------------------------
import os, glob

# CheXpert-small ships as CheXpert-v1.0-small/{train.csv,valid.csv} + image dirs.
# Kaggle mounts datasets under /kaggle/input/<dataset-slug>/... — auto-find it.
def find_chexpert_root():
    for base in glob.glob("/kaggle/input/**/train.csv", recursive=True):
        return os.path.dirname(base)
    raise FileNotFoundError("train.csv not found under /kaggle/input — attach the CheXpert-small dataset.")

DATA_ROOT   = find_chexpert_root()
INPUT_DIR   = "/kaggle/input"
WORK_DIR    = "/kaggle/working"
IMG_ROOT    = os.path.dirname(DATA_ROOT.rstrip("/"))  # images are referenced relative to here

INPUT_SIZE  = 224
BATCH_SIZE  = 32
EPOCHS      = 3            # bump once you confirm it runs; ~1 epoch to smoke-test
LR          = 1e-4
NUM_WORKERS = 2
SUBSET      = None         # e.g. 20000 to train on a subset first; None = full

# The 5 "competition" CheXpert labels most papers report AUROC on.
LABELS = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]

# Uncertainty (-1) mapping per label: "ones" (U-Ones) or "zeros" (U-Zeros).
# U-Ones tends to help Atelectasis/Edema; U-Zeros for the rest (Irvin et al. 2019).
UNCERTAIN_POLICY = {
    "Atelectasis": "ones", "Cardiomegaly": "zeros", "Consolidation": "zeros",
    "Edema": "ones", "Pleural Effusion": "zeros",
}

print("DATA_ROOT:", DATA_ROOT)
print("labels:", LABELS)

In [ ]:
# --- Data -------------------------------------------------------------------
import numpy as np, pandas as pd, torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

def load_split(csv_name):
    df = pd.read_csv(os.path.join(DATA_ROOT, csv_name))
    # Map uncertainty per label, fill blanks as 0 (negative).
    for lab in LABELS:
        col = df[lab].copy()
        policy = UNCERTAIN_POLICY.get(lab, "zeros")
        col = col.replace(-1.0, 1.0 if policy == "ones" else 0.0)
        df[lab] = col.fillna(0.0).clip(0, 1)
    return df

train_df = load_split("train.csv")
valid_df = load_split("valid.csv")
if SUBSET:
    train_df = train_df.sample(n=min(SUBSET, len(train_df)), random_state=0).reset_index(drop=True)
print("train:", len(train_df), "valid:", len(valid_df))

# Normalization must MATCH the Mac app's preprocessing (handoff contract).
# We use ImageNet normalization on 3-channel grayscale-repeated images.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class CheXpertDataset(Dataset):
    def __init__(self, df, tf):
        self.df, self.tf = df.reset_index(drop=True), tf
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        # Path column is relative like "CheXpert-v1.0-small/train/patient.../view1.jpg"
        img = Image.open(os.path.join(IMG_ROOT, row["Path"])).convert("RGB")
        y = torch.tensor([row[l] for l in LABELS], dtype=torch.float32)
        return self.tf(img), y

train_ds = CheXpertDataset(train_df, train_tf)
valid_ds = CheXpertDataset(valid_df, eval_tf)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
valid_dl = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

In [ ]:
# --- Model + training -------------------------------------------------------
from torchvision.models import densenet121, DenseNet121_Weights
from sklearn.metrics import roc_auc_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# Build EXACTLY like the Mac app's _build_densenet121 so the state_dict matches.
model = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)  # pretrained backbone
model.classifier = torch.nn.Linear(model.classifier.in_features, len(LABELS))
model = model.to(device)

criterion = torch.nn.BCEWithLogitsLoss()   # head outputs logits -> sigmoid at inference
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

@torch.no_grad()
def evaluate():
    model.eval()
    ys, ps = [], []
    for x, y in valid_dl:
        logits = model(x.to(device))
        ps.append(torch.sigmoid(logits).cpu().numpy())
        ys.append(y.numpy())
    y_true, y_prob = np.concatenate(ys), np.concatenate(ps)
    aucs = {}
    for i, lab in enumerate(LABELS):
        try:
            aucs[lab] = roc_auc_score(y_true[:, i], y_prob[:, i])
        except ValueError:
            aucs[lab] = float("nan")  # a label with one class present in valid
    return aucs

for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for step, (x, y) in enumerate(train_dl):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        running += loss.item()
        if step % 100 == 0:
            print(f"epoch {epoch} step {step}/{len(train_dl)} loss {loss.item():.4f}")
    aucs = evaluate()
    mean_auc = np.nanmean(list(aucs.values()))
    print(f"== epoch {epoch}: train_loss {running/len(train_dl):.4f} | mean AUROC {mean_auc:.4f}")
    for lab, a in aucs.items():
        print(f"     {lab:20s} {a:.4f}")

In [ ]:
# --- Export the handoff contract to /kaggle/working -------------------------
import json

out_dir = WORK_DIR  # download these three files after the run
torch.save(model.state_dict(), os.path.join(out_dir, "state_dict.pth"))

with open(os.path.join(out_dir, "labels.json"), "w") as f:
    json.dump(LABELS, f, indent=2)

model_config = {
    "arch": "densenet121",
    "input_size": INPUT_SIZE,
    "num_classes": len(LABELS),
    "norm": "imagenet",     # MUST match the Mac preprocess (ImageNet mean/std, 3ch)
    "sigmoid": True,        # head outputs logits; app applies sigmoid
    "uncertain_policy": UNCERTAIN_POLICY,
}
with open(os.path.join(out_dir, "model_config.json"), "w") as f:
    json.dump(model_config, f, indent=2)

print("Wrote to", out_dir, ":")
for fn in ["state_dict.pth", "labels.json", "model_config.json"]:
    p = os.path.join(out_dir, fn)
    print(f"  {fn:20s} {os.path.getsize(p)/1e6:.2f} MB")
print("\nDownload these three, put them in medical-ai/models/checkpoints/,")
print("then set MODEL_SOURCE=local on the Mac.")